# Notebook 2 - Evaluate all algorithms

Read JSON instances from `data/`, run every algorithm currently exposed by `MinimumCoverSet`, and save detailed results and plots under `results/`.

Each solver call runs in an isolated child process with a per-call timeout, so exact algorithms that are infeasible for large instances are recorded as `timeout` without killing the notebook kernel.


In [ ]:
from pathlib import Path
import json
import multiprocessing as mp
from queue import Empty
import sys
import time

import matplotlib.pyplot as plt
import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = Path(__file__).resolve().parents[1] if "__file__" in globals() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT / "src"))

from min_cover_set import MinimumCoverSet, SetCoverInstance
from min_cover_set.validation import validate_cover

DATA_DIR = REPO_ROOT / "data"
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
TIMEOUT_SECONDS = 120
RANDOMIZED_GREEDY_OPTIONS = {"seed": 0, "restarts": 30, "rcl_size": 3}


In [ ]:
def load_instance(path):
    payload = json.loads(path.read_text(encoding="utf-8"))
    instance = SetCoverInstance.from_collections(payload["universe"], payload["subsets"])
    return instance, payload.get("metadata", {})

def run_algorithm_in_child(instance, method_name, options, queue):
    try:
        selected = getattr(MinimumCoverSet(instance), method_name)(**options)
        validate_cover(instance, selected)
        queue.put({"status": "completed", "selected": len(selected), "solution": list(selected)})
    except Exception as error:
        queue.put({"status": "error", "error": f"{type(error).__name__}: {error}"})

def run_with_timeout(instance, method_name, options=None, timeout_seconds=TIMEOUT_SECONDS):
    options = options or {}
    context = mp.get_context("fork") if "fork" in mp.get_all_start_methods() else mp.get_context()
    queue = context.Queue()
    process = context.Process(target=run_algorithm_in_child, args=(instance, method_name, options, queue))
    started = time.perf_counter()
    process.start()
    process.join(timeout_seconds)
    elapsed_ms = (time.perf_counter() - started) * 1000
    if process.is_alive():
        process.kill()
        process.join()
        return {"status": "timeout", "runtime_ms": elapsed_ms, "selected": None, "solution": None}
    try:
        result = queue.get(timeout=1)
    except Empty:
        return {"status": "error", "runtime_ms": elapsed_ms, "selected": None, "solution": None, "error": "child process exited without a result"}
    result["runtime_ms"] = elapsed_ms
    result.setdefault("selected", None)
    result.setdefault("solution", None)
    return result

ALGORITHMS = [
    ("brute_force", "brute_force", {}),
    ("greedy", "greedy", {}),
    ("greedy_pruned", "greedy_pruned", {}),
    ("randomized_greedy", "randomized_greedy", RANDOMIZED_GREEDY_OPTIONS),
    ("dynamic_programming", "dynamic_programming", {}),
    ("branch_and_bound", "branch_and_bound", {}),
    ("binary_branch_and_bound", "binary_branch_and_bound", {}),
]

records = []
for path in sorted(DATA_DIR.glob("n*.json")):
    instance, metadata = load_instance(path)
    reference = None
    instance_records = []
    for name, method_name, options in ALGORITHMS:
        result = run_with_timeout(instance, method_name, options)
        if name == "brute_force" and result["status"] == "completed":
            reference = result["selected"]
        record = {"file": path.name, **metadata, "algorithm": name, "reference_optimal": reference, **result}
        instance_records.append(record)
    for record in instance_records:
        selected = record["selected"]
        record["accuracy"] = reference / selected if selected and reference else None
        record["is_optimal"] = selected == reference if selected is not None and reference is not None else None
        records.append(record)

results_df = pd.DataFrame(records)
results_df.to_json(RESULTS_DIR / "results.json", orient="records", indent=2)
results_df.to_csv(RESULTS_DIR / "results.csv", index=False)
print(f"Saved {len(results_df)} result rows to {RESULTS_DIR}")
print(results_df["status"].value_counts())
results_df.head()


In [ ]:
completed = results_df[results_df["status"] == "completed"]
summary = (
    completed.groupby(["num_elements", "density", "algorithm"], as_index=False)
    .agg(runtime_ms=("runtime_ms", "mean"), accuracy=("accuracy", "mean"), selected=("selected", "mean"))
)
display(summary.round({"runtime_ms": 3, "accuracy": 3, "selected": 2}))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
for algorithm, group in results_df.groupby("algorithm"):
    completed_group = group[group["status"] == "completed"]
    runtime = completed_group.groupby("num_elements", as_index=False)["runtime_ms"].mean()
    axes[0].plot(runtime["num_elements"], runtime["runtime_ms"], marker="o", label=algorithm)
    quality = completed_group.groupby("num_elements", as_index=False)["accuracy"].mean()
    axes[1].plot(quality["num_elements"], quality["accuracy"], marker="o", label=algorithm)
axes[0].set(title="Runtime by instance size", xlabel="Number of elements", ylabel="Milliseconds")
axes[1].set(title="Mean accuracy", xlabel="Number of elements", ylabel="Brute-force reference / selected size")
for axis in axes:
    axis.grid(True, alpha=0.3)
    axis.legend()
fig.tight_layout()
fig.savefig(RESULTS_DIR / "comparison.png", dpi=150)
plt.show()
print(f"Saved plot to {RESULTS_DIR / 'comparison.png'}")
